# Hari 24 — Retrain dengan Fitur Lag-0 & Bandingkan

**Benchmark lama yang harus dikalahkan (dari sebelum ada `lag_0`):**
- Linear Regression: MAE test 7.806,56 | Trend accuracy 62,07%
- Random Forest (tuned): MAE test 8.456,32 | Trend accuracy 31,03%

Hari ini kita latih ulang kedua model pakai `X_train_v2`/`X_test_v2` (18 fitur, termasuk `lag_0`) dan lihat apakah dugaan dari Hari 18 (fitur ini akan membantu) benar-benar terbukti.

In [25]:
# Cell ini sudah lengkap — muat data versi baru dan latih ulang kedua model,
# pakai hyperparameter Random Forest yang sama hasil tuning Hari 20 (max_depth=7,
# min_samples_leaf=2) supaya perbandingannya adil (yang berubah cuma fiturnya).

import pandas as pd
#import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train_v2 = pd.read_csv("X_train_v2.csv", index_col=0, parse_dates=True)
X_test_v2 = pd.read_csv("X_test_v2.csv", index_col=0, parse_dates=True)
X_train_scaled_v2 = pd.read_csv("X_train_scaled_v2.csv", index_col=0, parse_dates=True)
X_test_scaled_v2 = pd.read_csv("X_test_scaled_v2.csv", index_col=0, parse_dates=True)
y_train_v2 = pd.read_csv("y_train_v2.csv", index_col=0, parse_dates=True).iloc[:, 0]
y_test_v2 = pd.read_csv("y_test_v2.csv", index_col=0, parse_dates=True).iloc[:, 0]

model_linear_v2 = LinearRegression().fit(X_train_scaled_v2, y_train_v2)
model_rf_v2 = RandomForestRegressor(random_state=42, max_depth=7, min_samples_leaf=2).fit(X_train_v2, y_train_v2)

pred_linear_v2 = model_linear_v2.predict(X_test_scaled_v2)
pred_rf_v2 = model_rf_v2.predict(X_test_v2)

print("Kedua model v2 (dengan lag_0) berhasil dilatih.")

Kedua model v2 (dengan lag_0) berhasil dilatih.


## Bandingkan MAE: Versi Baru vs Versi Lama

In [26]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Hitung mae_linear_v2 = mean_absolute_error(y_test_v2, pred_linear_v2)
# 2. Hitung mae_rf_v2 = mean_absolute_error(y_test_v2, pred_rf_v2)
# 3. Cetak keduanya, lalu bandingkan LANGSUNG dengan angka lama:
#    - Linear Regression lama: 7806.56
#    - Random Forest tuned lama: 8456.32
#    Hitung selisih dan apakah membaik atau memburuk untuk masing-masing model
#    (hint: selisih = mae_lama - mae_baru; positif berarti membaik)
# Tulis kode kamu di bawah ini:
mae_linear_v2 = mean_absolute_error(y_test_v2, pred_linear_v2)
mae_rf_v2 = mean_absolute_error(y_test_v2, pred_rf_v2)
print(f"MAE Linear Regression V2: {mae_linear_v2:.2f}")
print(f"MAE Random Forest V2: {mae_rf_v2:.2f}")
mae_linear_lama = 7806.56
mae_rf_tuned_lama = 8456.32

selisih_lr_v2 = mae_linear_lama - mae_linear_v2
print(f"Selisih dengan Linear Regression lama: {selisih_lr_v2:.2f}")

selisih_rf_tuned_v2 = mae_rf_tuned_lama - mae_rf_v2
print(f"Selisih dengan Random Forest (tuned) lama: {selisih_rf_tuned_v2:.2f}")

print()
if selisih_lr_v2 > 0:
    print("Linear Regression membaik")
else:
    print("Linear Regression tidak membaik")

if selisih_rf_tuned_v2 > 0:
    print("Random Forest membaik")
else:
    print("Random Forest tidak membaik")

MAE Linear Regression V2: 7806.56
MAE Random Forest V2: 5913.70
Selisih dengan Linear Regression lama: 0.00
Selisih dengan Random Forest (tuned) lama: 2542.62

Linear Regression membaik
Random Forest membaik


## Cek Koefisien & Feature Importance dengan Lag-0

In [27]:
# Cell ini sudah lengkap.
koefisien_v2 = pd.Series(model_linear_v2.coef_, index=X_train_scaled_v2.columns)
print("Koefisien Linear Regression (v2), diurutkan:\n")
print(koefisien_v2.reindex(koefisien_v2.abs().sort_values(ascending=False).index).head(6))

importance_v2 = pd.Series(model_rf_v2.feature_importances_, index=X_train_v2.columns)
print("\n\nFeature importance Random Forest (v2), diurutkan:\n")
print(importance_v2.sort_values(ascending=False).head(6))

Koefisien Linear Regression (v2), diurutkan:

lag_0              118349.630169
lag_1              -49894.985436
lag_2              -21141.549425
rolling_mean_4w     14618.107068
bulan_1             11810.248311
bulan_7             11211.833196
dtype: float64


Feature importance Random Forest (v2), diurutkan:

lag_0              0.894742
lag_3              0.035053
lag_2              0.022495
lag_1              0.016990
rolling_mean_4w    0.011145
vaksin_persen      0.007763
dtype: float64


In [28]:
# 1. Pastikan lag_0 di X_train_scaled_v2 memang punya variasi nyata (bukan konstan/rusak)
print(X_train_scaled_v2["lag_0"].describe())

# 2. Bandingkan barisan prediksi baru vs simpan salinan prediksi lama untuk dicek manual
#    (kalau kamu masih punya notebook Hari 16/17, bandingkan 5 nilai pertama pred_test_linear
#    dengan pred_linear_v2[:5] — apakah benar-benar identik atau cuma mirip?)
print(pred_linear_v2[:5])

# 3. Cek urutan kolom saat fit vs saat tabel koefisien dicetak — potensi mismatch label
print(X_train_scaled_v2.columns.tolist())

count    1.140000e+02
mean     1.032313e-16
std      1.004415e+00
min     -6.525152e-01
25%     -5.958580e-01
50%     -3.241948e-01
75%     -2.422020e-02
max      4.203358e+00
Name: lag_0, dtype: float64
[20702.8233789  22304.20606571 27396.69133955 32271.95864653
 33281.16821714]
['lag_0', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_4w', 'vaksin_persen', 'bulan_1', 'bulan_2', 'bulan_3', 'bulan_4', 'bulan_5', 'bulan_6', 'bulan_7', 'bulan_8', 'bulan_9', 'bulan_10', 'bulan_11', 'bulan_12']


In [31]:
print(X_test_scaled_v2["lag_0"].describe())

count    29.000000
mean     -0.380088
std       0.155775
min      -0.632365
25%      -0.507634
50%      -0.416685
75%      -0.252865
max      -0.077085
Name: lag_0, dtype: float64


**Yang perlu diperhatikan:** apakah `lag_0` langsung jadi fitur paling dominan di Random Forest (kemungkinan besar iya, karena dia adalah info paling terbaru dan paling relevan)? Dan seperti biasa, koefisien Linear Regression individual di sini **tetap harus dibaca hati-hati** — sekarang multikolinearitasnya malah lebih parah dari sebelumnya (korelasi lag_0-lag_1 di Hari 23 = 0,9279).

## Trend Accuracy dengan Lag-0

Ada bonus praktis dari fitur `lag_0`: kolom ini **PERSIS SAMA** dengan `current_actual_test` yang harus direkonstruksi susah payah dari `dataset_bersih_minggu2.csv` di Hari 18 dan 21-22. Sekarang tinggal pakai `X_test_v2["lag_0"]` langsung — tidak perlu load file terpisah lagi.

In [29]:
# Cell ini sudah lengkap.
def label_tren(nilai_depan, nilai_sekarang, ambang=0.05):
    perubahan = (nilai_depan - nilai_sekarang) / nilai_sekarang
    if perubahan > ambang:
        return "Naik"
    elif perubahan < -ambang:
        return "Turun"
    else:
        return "Stabil"

current_actual_v2 = X_test_v2["lag_0"]
tren_aktual_v2 = [label_tren(y_test_v2.iloc[i], current_actual_v2.iloc[i]) for i in range(len(y_test_v2))]

In [30]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat tren_prediksi_linear_v2: list label_tren(pred_linear_v2[i], current_actual_v2.iloc[i])
#    untuk i dalam range(len(pred_linear_v2))
# 2. Buat tren_prediksi_rf_v2 dengan cara yang sama, pakai pred_rf_v2
# 3. Hitung trend_accuracy_linear_v2 dan trend_accuracy_rf_v2 (proporsi yang sama dengan
#    tren_aktual_v2), sama seperti pola Hari 18/21
# 4. Cetak keduanya, bandingkan LANGSUNG dengan angka lama:
#    - Linear Regression lama: 62.07%
#    - Random Forest tuned lama: 31.03%
# Tulis kode kamu di bawah ini:
tren_prediksi_linear_v2 = [label_tren(pred_linear_v2[i], current_actual_v2.iloc[i]) 
                           for i in range(len(pred_linear_v2))]
tren_prediksi_rf_v2 = [label_tren(pred_rf_v2[i], current_actual_v2.iloc[i]) 
                       for i in range(len(pred_rf_v2))]
trend_accuracy_linear_v2 = (pd.Series(tren_prediksi_linear_v2) == pd.Series(tren_aktual_v2)).mean()
trend_accuracy_rf_v2 = (pd.Series(tren_prediksi_rf_v2) == pd.Series(tren_aktual_v2)).mean()

akurasi_lr_lama = 62.07
akurasi_rf_tuned_lama = 31.03
print(f"Akurasi Linear Regression lama: {akurasi_lr_lama}%")
print(f"Akurasi Random Forest (tuned) lama: {akurasi_rf_tuned_lama}%")

print(f"Akurasi Linear Regression baru: {trend_accuracy_linear_v2*100:.2f}%")
print(f"Akurasi Random Forest (tuned) baru: {trend_accuracy_rf_v2*100:.2f}%")

Akurasi Linear Regression lama: 62.07%
Akurasi Random Forest (tuned) lama: 31.03%
Akurasi Linear Regression baru: 62.07%
Akurasi Random Forest (tuned) baru: 48.28%


## Refleksi Hari 24

> 1. Apakah MAE test kedua model membaik dengan `lag_0`? Model mana yang paling banyak terbantu? → **hanya Random Forest tuned**
> 2. Apakah `lag_0` benar-benar jadi fitur paling penting di Random Forest, sesuai dugaan? → **Ya**
> 3. Apakah trend accuracy juga membaik? Apakah polanya sama dengan MAE, atau ada model yang MAE-nya membaik tapi trend accuracy-nya malah memburuk (atau sebaliknya)? → **Trend accuracy Random Forest tuned memabaik**
> 4. Berdasarkan hasil single-split ini SAJA — apakah iterasi menambah `lag_0` ini terlihat berhasil? (Ingat: kita belum tahu jawaban yang lebih meyakinkan sampai dicek pakai cross-validation adil seperti Hari 22, itu agenda besok) → **belum tahu, tapi sejauh ini untuk model random forest tuned terlihat berhasil**

---
### Selanjutnya: Hari 25 — Evaluation Ronde Kedua (Final)

Ulangi metodologi "Uji Kewajaran" dari Hari 22 (perbandingan fold-demi-fold yang adil terhadap baseline), tapi kali ini pakai `X_full_v2`/`y_full_v2`. Ini keputusan Go/No-Go **final** — hasil apa pun yang keluar, kita dokumentasikan dan lanjut ke Deployment (tidak ada iterasi ketiga, supaya proyek selesai dalam 30 hari).